# arXiv HTML Reference + Jaccard vs PyMuPDF

Notebook này làm 2 việc:

1. **Fetch + parse HTML full-text từ arXiv** để tạo `arxiv_html_reference.jsonl`
2. **Đọc `pymupdf.jsonl` có sẵn** rồi tính **word-level Jaccard** giữa `arxiv_html` và `pymupdf` trên cùng `doc_id`

Mục tiêu là kiểm tra xem text HTML bạn lấy ra có **đủ sạch** và **đủ tương đồng** với text từ PDF parser (`PyMuPDF`) hay không.

## Input / Output chính

### Input
- `arxiv_ocr_benchmark_workspace/metadata/arxiv_search_results_multi_query.jsonl`
- `arxiv_ocr_benchmark_workspace/parsed_jsonl/pymupdf.jsonl`

### Output
- `arxiv_html_reference_workspace/parsed_jsonl/arxiv_html_reference.jsonl`
- `arxiv_html_reference_workspace/metadata/arxiv_html_reference_meta.csv`
- `arxiv_html_reference_workspace/jaccard/arxiv_html_vs_pymupdf_jaccard.csv`

## Ý tưởng làm sạch HTML
Bản này giữ logic đường dẫn/workspace của notebook cũ, nhưng chỉnh parser để:
- ưu tiên đúng vùng nội dung paper (`article`, `section.ltx_document`, ...)
- xóa bớt boilerplate / nav / footer / bibliography / footnote / math / figure
- **không fallback quá sớm về `body`**
- chuẩn hóa whitespace vừa phải để hạn chế nhiễu


In [1]:
%pip -q install requests beautifulsoup4 lxml tqdm pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


## 1. Import và cấu hình

In [2]:
from pathlib import Path
import re
import json
import time
import random
import hashlib
import threading
import concurrent.futures
from collections import Counter

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

random.seed(42)

# ── Base paths: giữ logic giống notebook cũ ──────────────────────────────────
_BASE = Path("notebook") if Path("notebook").exists() else Path(".")
_ABS  = Path(r"d:\Project\LSHBloomExperiments\notebook")

ROOT = _ABS if _ABS.exists() else _BASE
print(f"ROOT = {ROOT.resolve()}")

# ── Input metadata + existing PyMuPDF output ─────────────────────────────────
PREV_META_DIR = ROOT / "arxiv_ocr_benchmark_workspace" / "metadata"
META_JSONL    = PREV_META_DIR / "arxiv_search_results_multi_query.jsonl"
META_CSV      = PREV_META_DIR / "arxiv_search_results_multi_query.csv"

PYMUPDF_JSONL = ROOT / "arxiv_ocr_benchmark_workspace" / "parsed_jsonl" / "pymupdf.jsonl"

# ── Output dirs ───────────────────────────────────────────────────────────────
BASE_DIR   = ROOT / "arxiv_html_reference_workspace"
HTML_DIR   = BASE_DIR / "html_raw"
TEXT_DIR   = BASE_DIR / "html_text"
PARSED_DIR = BASE_DIR / "parsed_jsonl"
META_DIR   = BASE_DIR / "metadata"
JACC_DIR   = BASE_DIR / "jaccard"

for d in [HTML_DIR, TEXT_DIR, PARSED_DIR, META_DIR, JACC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Fetch config ──────────────────────────────────────────────────────────────
NUM_WORKERS = 4
SLEEP_SEC   = 1.5
MAX_RETRIES = 5
TIMEOUT     = 30
MIN_CHARS   = 300

print(f"Metadata JSONL : {META_JSONL}  exists={META_JSONL.exists()}")
print(f"PyMuPDF JSONL  : {PYMUPDF_JSONL}  exists={PYMUPDF_JSONL.exists()}")
print(f"Output dir     : {BASE_DIR}")


ROOT = D:\Project\LSHBloomExperiments\notebook
Metadata JSONL : d:\Project\LSHBloomExperiments\notebook\arxiv_ocr_benchmark_workspace\metadata\arxiv_search_results_multi_query.jsonl  exists=True
PyMuPDF JSONL  : d:\Project\LSHBloomExperiments\notebook\arxiv_ocr_benchmark_workspace\parsed_jsonl\pymupdf.jsonl  exists=True
Output dir     : d:\Project\LSHBloomExperiments\notebook\arxiv_html_reference_workspace


c:\Users\pntha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load metadata và PyMuPDF corpus

In [3]:
# ── Load metadata ───────────────────────────────────────────────────────────
if META_JSONL.exists():
    rows = []
    with open(META_JSONL, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    df_meta = pd.DataFrame(rows)
    print(f"Loaded {len(df_meta):,} metadata rows from JSONL")
elif META_CSV.exists():
    df_meta = pd.read_csv(META_CSV)
    print(f"Loaded {len(df_meta):,} metadata rows from CSV")
else:
    raise FileNotFoundError(
        f"Không tìm thấy metadata.\n"
        f"Expected one of:\n  {META_JSONL}\n  {META_CSV}"
    )

def build_html_url(arxiv_id: str) -> str:
    return f"https://arxiv.org/html/{arxiv_id}"

df_meta["html_url"] = df_meta["arxiv_id"].astype(str).apply(build_html_url)

# ── Load existing PyMuPDF parsed text ─────────────────────────────────────────
if not PYMUPDF_JSONL.exists():
    raise FileNotFoundError(f"Không tìm thấy file PyMuPDF JSONL: {PYMUPDF_JSONL}")

pymupdf_map = {}
with open(PYMUPDF_JSONL, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        d = json.loads(line)
        doc_id = str(d.get("doc_id", ""))
        text   = d.get("text") or ""
        if doc_id and len(text) >= MIN_CHARS:
            pymupdf_map[doc_id] = text

print(f"Loaded PyMuPDF valid docs: {len(pymupdf_map):,}")

# Ưu tiên chỉ fetch những doc_id đã có PyMuPDF để so Jaccard luôn
df_work = df_meta[df_meta["arxiv_id"].astype(str).isin(pymupdf_map.keys())].copy()
print(f"Docs to fetch (intersect with PyMuPDF corpus): {len(df_work):,}")

display(df_work[["arxiv_id", "title", "primary_category", "published", "html_url"]].head(5))


Loaded 1,579 metadata rows from JSONL
Loaded PyMuPDF valid docs: 1,578
Docs to fetch (intersect with PyMuPDF corpus): 1,578


,arxiv_id,title,primary_category,published,html_url
0,2604.01212v1,$\texttt{YC-Bench}$: Benchmarking AI Agents fo...,cs.CL,2026-04-01T17:52:19Z,https://arxiv.org/html/2604.01212v1
1,2604.01195v1,ORBIT: Scalable and Verifiable Data Generation...,cs.CL,2026-04-01T17:42:41Z,https://arxiv.org/html/2604.01195v1
2,2604.01193v1,Embarrassingly Simple Self-Distillation Improv...,cs.CL,2026-04-01T17:39:50Z,https://arxiv.org/html/2604.01193v1
3,2604.01181v1,True (VIS) Lies: Analyzing How Generative AI R...,cs.HC,2026-04-01T17:33:10Z,https://arxiv.org/html/2604.01181v1
4,2604.01029v1,Revision or Re-Solving? Decomposing Second-Pas...,cs.SE,2026-04-01T15:39:40Z,https://arxiv.org/html/2604.01029v1


## 3. HTML fetcher và parser sạch hơn

In [4]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; arxiv-html-crawler/1.0; research-only)",
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "en-US,en;q=0.9",
})

def fetch_html(url: str, timeout: int = TIMEOUT, max_retries: int = MAX_RETRIES):
    last_status = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = SESSION.get(url, timeout=timeout, allow_redirects=True)
            last_status = resp.status_code
            if resp.status_code == 200:
                return resp.text, 200
            if resp.status_code == 404:
                return None, 404
            if resp.status_code in (429, 500, 502, 503, 504):
                wait = (2 ** attempt) + random.uniform(0, 1.0)
                print(f"  [HTTP {resp.status_code}] {url[:90]} → wait {wait:.1f}s ({attempt}/{max_retries})")
                time.sleep(wait)
                continue
            return None, resp.status_code
        except requests.exceptions.Timeout:
            wait = (2 ** attempt) + random.uniform(0, 1.0)
            print(f"  [Timeout] {url[:90]} → wait {wait:.1f}s ({attempt}/{max_retries})")
            time.sleep(wait)
        except requests.exceptions.RequestException as e:
            wait = (2 ** attempt) + random.uniform(0, 1.0)
            print(f"  [Error] {e} → wait {wait:.1f}s ({attempt}/{max_retries})")
            time.sleep(wait)
    return None, last_status

# Tag loại bỏ hoàn toàn
_REMOVE_TAGS = [
    "script", "style", "noscript", "nav", "footer", "header", "aside",
    "svg", "canvas", "form", "button", "input", "select", "textarea",
    "math", "figure", "figcaption",
]

# Selector vùng paper ưu tiên
_CONTENT_SELECTORS = [
    "article",
    "section.ltx_document",
    "div.ltx_document",
    "div.ltx_page_content",
    "main",
    "div#content",
]

# Selector nhiễu thường gặp trong arXiv/LaTeXML
_DROP_SELECTORS = [
    ".ltx_page_header", ".ltx_page_footer",
    ".ltx_bibliography", ".ltx_biblist", ".ltx_toclist",
    ".ltx_role_footnote", ".ltx_note_outer", ".ltx_note_mark",
    ".ltx_authors", ".ltx_dates",
    "[role='navigation']",
    "#references", "#bibliography", "#footnotes",
]

def _pick_content_node(soup: BeautifulSoup):
    for selector in _CONTENT_SELECTORS:
        node = soup.select_one(selector)
        if node:
            txt = node.get_text(" ", strip=True)
            if len(txt) > 500:
                return node, selector
    body = soup.body
    if body and len(body.get_text(" ", strip=True)) > 1000:
        return body, "body_fallback"
    return soup, "soup_fallback"

def _normalize_text(text: str) -> str:
    text = re.sub(r"\r\n?", "\n", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # bỏ các dòng quá ngắn chỉ chứa marker/ký hiệu vụn
    lines = []
    for ln in text.split("\n"):
        s = ln.strip()
        if not s:
            lines.append("")
            continue
        if re.fullmatch(r"[\[\]\(\)\{\}\d,\.\-\s]{1,6}", s):
            continue
        if re.fullmatch(r"[\W_]{1,6}", s):
            continue
        lines.append(s)

    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def parse_html_to_text(html: str, return_selector: bool = False):
    soup = BeautifulSoup(html, "lxml")

    for tag_name in _REMOVE_TAGS:
        for tag in soup.find_all(tag_name):
            tag.decompose()

    content_node, chosen_selector = _pick_content_node(soup)

    for sel in _DROP_SELECTORS:
        for node in content_node.select(sel):
            node.decompose()

    text = content_node.get_text(separator="\n", strip=False)
    text = _normalize_text(text)

    if return_selector:
        return text, chosen_selector
    return text

def sha1_text(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()

print("Parser ready.")
print("Content selectors:", _CONTENT_SELECTORS)


Parser ready.
Content selectors: ['article', 'section.ltx_document', 'div.ltx_document', 'div.ltx_page_content', 'main', 'div#content']


## 4. Test thử 1 bài

In [5]:
sample = df_work.iloc[0]
print(f"Doc  : {sample['arxiv_id']}")
print(f"Title: {sample['title'][:100]}")
print(f"URL  : {sample['html_url']}")

html_raw, status = fetch_html(sample["html_url"])
if html_raw:
    text, selector = parse_html_to_text(html_raw, return_selector=True)
    print(f"\n✅ Fetch OK (HTTP {status})")
    print(f"Chosen selector : {selector}")
    print(f"HTML bytes      : {len(html_raw):,}")
    print(f"Text chars      : {len(text):,}")
    print("\n--- Text preview (800 chars) ---")
    print(text[:800])
else:
    print(f"❌ Fetch failed: HTTP {status}")


Doc  : 2604.01212v1
Title: $\texttt{YC-Bench}$: Benchmarking AI Agents for Long-Term Planning and Consistent Execution
URL  : https://arxiv.org/html/2604.01212v1

✅ Fetch OK (HTTP 200)
Chosen selector : article
HTML bytes      : 279,573
Text chars      : 30,881

--- Text preview (800 chars) ---
\ycbench
: Benchmarking AI Agents for Long-Term Planning and Consistent Execution

Abstract

As LLM agents tackle increasingly complex tasks, a critical question is whether they can maintain strategic coherence over long horizons: planning under uncertainty, learning from delayed feedback, and adapting when early mistakes compound. We introduce
\ycbench
, a benchmark that evaluates these capabilities by tasking an agent with running a simulated startup over a one-year horizon spanning hundreds of turns. The agent must manage employees, select task contracts, and maintain profitability in a partially observable environment where adversarial clients and growing payroll create compounding consequen

## 5. Fetch + parse toàn bộ HTML (resume-safe)

In [6]:
RESUME_PATH = PARSED_DIR / "arxiv_html_reference.jsonl"
_lock = threading.Lock()

done_ids = set()
valid_records = []

if RESUME_PATH.exists():
    with open(RESUME_PATH, encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
                done_ids.add(str(rec["doc_id"]))
                valid_records.append(rec)
            except Exception:
                pass

    # rewrite file sạch
    with open(RESUME_PATH, "w", encoding="utf-8") as f:
        for r in valid_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Resume loaded: {len(done_ids):,} done docs")

records = list(valid_records)
errors  = []

todo_df = df_work[~df_work["arxiv_id"].astype(str).isin(done_ids)].copy()
print(f"To fetch now: {len(todo_df):,}")

def process_one(row: dict):
    arxiv_id = str(row.get("arxiv_id", ""))
    html_url = str(row.get("html_url", ""))
    title    = str(row.get("title", ""))
    cat      = str(row.get("primary_category", ""))
    pub      = str(row.get("published", ""))
    pdf_url  = str(row.get("pdf_url", ""))

    safe_id   = re.sub(r"[^\w.\-]", "_", arxiv_id)[:120]
    html_path = HTML_DIR / f"{safe_id}.html"
    text_path = TEXT_DIR / f"{safe_id}.txt"

    html_raw, status = fetch_html(html_url)
    time.sleep(SLEEP_SEC + random.uniform(0, 0.5))

    if html_raw is None:
        rec = {
            "doc_id": arxiv_id,
            "parser_name": "arxiv_html",
            "source_type": "html_reference",
            "title": title,
            "primary_category": cat,
            "published": pub,
            "html_url": html_url,
            "pdf_url": pdf_url,
            "html_path": "",
            "text_path": "",
            "char_count": 0,
            "line_count": 0,
            "text_sha1": "",
            "text": "",
            "fetch_status": f"failed_{status}",
            "metadata": {
                "source": "arxiv",
                "http_status": status,
                "html_available": False,
                "summary": str(row.get("summary", "")),
                "authors": row.get("authors", []),
                "categories": row.get("categories", []),
                "selector_used": None,
            },
        }
        return rec, None

    html_path.write_text(html_raw, encoding="utf-8", errors="replace")

    try:
        text, selector = parse_html_to_text(html_raw, return_selector=True)
    except Exception as e:
        return None, {"doc_id": arxiv_id, "error": f"parse error: {e}"}

    text_path.write_text(text, encoding="utf-8", errors="replace")
    is_valid = len(text) >= MIN_CHARS

    rec = {
        "doc_id": arxiv_id,
        "parser_name": "arxiv_html",
        "source_type": "html_reference",
        "title": title,
        "primary_category": cat,
        "published": pub,
        "html_url": html_url,
        "pdf_url": pdf_url,
        "html_path": str(html_path),
        "text_path": str(text_path),
        "char_count": len(text),
        "line_count": text.count("\n") + 1,
        "text_sha1": sha1_text(text),
        "text": text,
        "fetch_status": "ok" if is_valid else "empty",
        "metadata": {
            "source": "arxiv",
            "http_status": 200,
            "html_available": is_valid,
            "summary": str(row.get("summary", "")),
            "authors": row.get("authors", []),
            "categories": row.get("categories", []),
            "selector_used": selector,
        },
    }
    return rec, None

todo_rows = todo_df.to_dict(orient="records")
newly_done = 0

out_f = open(RESUME_PATH, "a", encoding="utf-8", errors="replace")
pbar  = tqdm(total=len(todo_rows), desc="Fetch HTML")

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    future_map = {executor.submit(process_one, row): row for row in todo_rows}
    for future in concurrent.futures.as_completed(future_map):
        rec, err = future.result()
        with _lock:
            if rec:
                records.append(rec)
                newly_done += 1
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                out_f.flush()
            if err:
                errors.append(err)
        pbar.update(1)

pbar.close()
out_f.close()

ok_count    = sum(1 for r in records if r["fetch_status"] == "ok")
empty_count = sum(1 for r in records if r["fetch_status"] == "empty")
fail_count  = sum(1 for r in records if str(r["fetch_status"]).startswith("failed"))

print(f"\nTotal records : {len(records):,}")
print(f"OK            : {ok_count:,}")
print(f"Empty         : {empty_count:,}")
print(f"Failed        : {fail_count:,}")
print(f"Parse errors  : {len(errors):,}")
if errors:
    print("First parse error:", errors[0])


Resume loaded: 1,579 done docs
To fetch now: 0


Fetch HTML: 0it [00:00, ?it/s]


Total records : 1,579
OK            : 1,421
Empty         : 1
Failed        : 157
Parse errors  : 0


## 6. Thống kê nhanh HTML output

In [7]:
stats_df = pd.DataFrame([
    {
        "doc_id": r["doc_id"],
        "primary_category": r.get("primary_category", ""),
        "published": str(r.get("published", ""))[:10],
        "char_count": r.get("char_count", 0),
        "line_count": r.get("line_count", 0),
        "fetch_status": r.get("fetch_status", ""),
        "selector_used": (r.get("metadata") or {}).get("selector_used"),
    }
    for r in records
])

ok_df = stats_df[stats_df["fetch_status"] == "ok"].copy()

print(f"Total: {len(stats_df):,}")
print("\nfetch_status:")
print(stats_df["fetch_status"].value_counts().to_string())

if len(ok_df):
    print("\nselector_used:")
    print(ok_df["selector_used"].fillna("NA").value_counts().to_string())

    print("\nchar_count describe:")
    print(ok_df["char_count"].describe().to_string())

    display(ok_df.head(10))


Total: 1,579

fetch_status:
fetch_status
ok            1421
failed_404     157
empty            1

selector_used:
selector_used
NA    1421

char_count describe:
count      1421.000000
mean      50044.662210
std       31601.347455
min        5552.000000
25%       33294.000000
50%       44865.000000
75%       59327.000000
max      645059.000000


,doc_id,primary_category,published,char_count,line_count,fetch_status,selector_used
0,2604.01212v1,cs.CL,2026-04-01,31258,538,ok,None
2,2604.01181v1,cs.HC,2026-04-01,67623,1761,ok,None
3,2604.01193v1,cs.CL,2026-04-01,106308,2659,ok,None
4,2604.01029v1,cs.SE,2026-04-01,32828,720,ok,None
5,2604.00986v1,cs.CR,2026-04-01,37030,1204,ok,None
6,2604.00890v1,cs.AI,2026-04-01,43228,1677,ok,None
7,2604.00913v1,cs.CV,2026-04-01,53634,1103,ok,None
8,2604.00778v1,cs.CL,2026-04-01,69129,1724,ok,None
9,2604.00698v1,cs.LG,2026-04-01,38383,1518,ok,None
10,2604.00455v1,cs.CV,2026-04-01,49486,1956,ok,None


## 7. Tính Jaccard `arxiv_html vs pymupdf`

In [8]:
def word_jaccard(ta: str, tb: str) -> float:
    sa = set(str(ta).lower().split())
    sb = set(str(tb).lower().split())
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

# Build HTML map từ records đã fetch
html_map = {
    str(r["doc_id"]): r["text"]
    for r in records
    if r.get("fetch_status") == "ok" and len(r.get("text", "")) >= MIN_CHARS
}

common_ids = sorted(set(html_map.keys()) & set(pymupdf_map.keys()))
print(f"HTML valid docs     : {len(html_map):,}")
print(f"PyMuPDF valid docs  : {len(pymupdf_map):,}")
print(f"Common valid docs   : {len(common_ids):,}")

rows = []
for doc_id in tqdm(common_ids, desc="Computing Jaccard"):
    html_text = html_map[doc_id]
    pdf_text  = pymupdf_map[doc_id]
    jac = word_jaccard(html_text, pdf_text)

    rows.append({
        "doc_id": doc_id,
        "jaccard_word_unigram": jac,
        "html_chars": len(html_text),
        "pymupdf_chars": len(pdf_text),
        "char_ratio_html_over_pdf": round(len(html_text) / max(1, len(pdf_text)), 4),
    })

jacc_df = pd.DataFrame(rows).sort_values("jaccard_word_unigram").reset_index(drop=True)
display(jacc_df.head(10))

if len(jacc_df):
    print("\n=== arxiv_html vs pymupdf — Jaccard summary ===")
    print(f"n      = {len(jacc_df):,}")
    print(f"mean   = {jacc_df['jaccard_word_unigram'].mean():.4f}")
    print(f"median = {jacc_df['jaccard_word_unigram'].median():.4f}")
    print(f"p10    = {jacc_df['jaccard_word_unigram'].quantile(0.10):.4f}")
    print(f"p90    = {jacc_df['jaccard_word_unigram'].quantile(0.90):.4f}")


HTML valid docs     : 1,421
PyMuPDF valid docs  : 1,578
Common valid docs   : 1,421


Computing Jaccard: 100%|██████████| 1421/1421 [00:13<00:00, 105.97it/s]


,doc_id,jaccard_word_unigram,html_chars,pymupdf_chars,char_ratio_html_over_pdf
0,2603.26221v1,0.090611,6015,115413,0.0521
1,2603.27146v1,0.131651,12514,100057,0.1251
2,2603.28651v1,0.170504,14044,107030,0.1312
3,2603.21577v1,0.181913,9154,43918,0.2084
4,2603.21481v1,0.187336,15722,73557,0.2137
5,2603.28618v1,0.188263,14545,77098,0.1887
6,2602.22220v1,0.188482,19647,124668,0.1576
7,2603.29109v1,0.210304,11881,64233,0.1850
8,2601.03531v1,0.216227,20395,126322,0.1615
9,2604.00547v1,0.219003,22616,97496,0.2320



=== arxiv_html vs pymupdf — Jaccard summary ===
n      = 1,421
mean   = 0.6011
median = 0.6060
p10    = 0.4611
p90    = 0.7406


## 8. Inspect vài case thấp / cao

In [9]:
if len(jacc_df) == 0:
    print("Chưa có common docs để inspect.")
else:
    print("=== Lowest Jaccard cases ===")
    display(jacc_df.head(10))

    print("\n=== Highest Jaccard cases ===")
    display(jacc_df.tail(10).sort_values("jaccard_word_unigram", ascending=False))

    low_ids = jacc_df.head(min(3, len(jacc_df)))["doc_id"].tolist()
    for doc_id in low_ids:
        print("=" * 100)
        print(f"doc_id  : {doc_id}")
        print(f"jaccard : {jacc_df.loc[jacc_df['doc_id'] == doc_id, 'jaccard_word_unigram'].iloc[0]:.4f}")
        print("\n--- HTML preview ---")
        print(html_map[doc_id][:1200])
        print("\n--- PyMuPDF preview ---")
        print(pymupdf_map[doc_id][:1200])
        print()


=== Lowest Jaccard cases ===


,doc_id,jaccard_word_unigram,html_chars,pymupdf_chars,char_ratio_html_over_pdf
0,2603.26221v1,0.090611,6015,115413,0.0521
1,2603.27146v1,0.131651,12514,100057,0.1251
2,2603.28651v1,0.170504,14044,107030,0.1312
3,2603.21577v1,0.181913,9154,43918,0.2084
4,2603.21481v1,0.187336,15722,73557,0.2137
5,2603.28618v1,0.188263,14545,77098,0.1887
6,2602.22220v1,0.188482,19647,124668,0.1576
7,2603.29109v1,0.210304,11881,64233,0.1850
8,2601.03531v1,0.216227,20395,126322,0.1615
9,2604.00547v1,0.219003,22616,97496,0.2320



=== Highest Jaccard cases ===


,doc_id,jaccard_word_unigram,html_chars,pymupdf_chars,char_ratio_html_over_pdf
1420,2603.14170v1,0.948466,63208,64407,0.9814
1419,2603.21745v1,0.935280,95334,95058,1.0029
1418,2603.14173v1,0.910642,42267,44300,0.9541
1417,2603.14629v1,0.899657,17953,19595,0.9162
1416,2510.24402v1,0.897059,89958,90158,0.9978
1415,2603.25821v1,0.881505,74712,77625,0.9625
1414,2604.00716v1,0.879504,30459,31766,0.9589
1413,2511.01386v1,0.873356,143619,148004,0.9704
1412,2603.21301v1,0.871257,17306,17820,0.9712
1411,2603.27435v1,0.870175,137472,159132,0.8639


doc_id  : 2603.26221v1
jaccard : 0.0906

--- HTML preview ---
Clawed
and Dangerous:

Can We Trust Open Agentic Systems?

Shiping Chen
1
, Qin Wang
1,2
, Guangsheng Yu
2
, Xu Wang
2
, Liming Zhu
1

1
CSIRO Data61

2
University of Technology Sydney

Abstract.

Open agentic systems combine LLM-based planning with external capabilities, persistent memory, and privileged execution. They are used in coding assistants, browser copilots, and enterprise automation. OpenClaw
 is a visible instance of this broader class.

Without much attention yet, their security challenge is fundamentally different from that of traditional software that relies on predictable execution and well-defined control flow. In open agentic systems, everything is “probabilistic”: plans are generated at runtime, key decisions may be shaped by untrusted natural-language inputs and tool outputs, execution unfolds in uncertain environments, and actions are taken under authority delegated by human users. The central challenge

## 9. Save kết quả

In [10]:
# Save metadata
META_COLS = [
    "doc_id", "parser_name", "source_type", "title",
    "primary_category", "published", "html_url", "pdf_url",
    "html_path", "text_path", "char_count", "line_count",
    "text_sha1", "fetch_status"
]

meta_rows = [{k: r.get(k, "") for k in META_COLS} for r in records]
meta_df = pd.DataFrame(meta_rows)

csv_path   = META_DIR / "arxiv_html_reference_meta.csv"
jsonl_path = META_DIR / "arxiv_html_reference_meta.jsonl"

meta_df.to_csv(csv_path, index=False, encoding="utf-8")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in meta_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Save jaccard
jacc_csv = JACC_DIR / "arxiv_html_vs_pymupdf_jaccard.csv"
jacc_df.to_csv(jacc_csv, index=False, encoding="utf-8")

summary = {
    "n_html_ok": int(sum(1 for r in records if r.get("fetch_status") == "ok")),
    "n_pymupdf_ok": int(len(pymupdf_map)),
    "n_common_valid": int(len(common_ids)),
    "mean_jaccard_html_vs_pymupdf": float(jacc_df["jaccard_word_unigram"].mean()) if len(jacc_df) else None,
    "median_jaccard_html_vs_pymupdf": float(jacc_df["jaccard_word_unigram"].median()) if len(jacc_df) else None,
    "p10_jaccard_html_vs_pymupdf": float(jacc_df["jaccard_word_unigram"].quantile(0.10)) if len(jacc_df) else None,
    "p90_jaccard_html_vs_pymupdf": float(jacc_df["jaccard_word_unigram"].quantile(0.90)) if len(jacc_df) else None,
}
summary_path = JACC_DIR / "arxiv_html_vs_pymupdf_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved:")
print(" ", RESUME_PATH)
print(" ", csv_path)
print(" ", jsonl_path)
print(" ", jacc_csv)
print(" ", summary_path)
print("\nSummary:")
print(json.dumps(summary, indent=2, ensure_ascii=False))


Saved:
  d:\Project\LSHBloomExperiments\notebook\arxiv_html_reference_workspace\parsed_jsonl\arxiv_html_reference.jsonl
  d:\Project\LSHBloomExperiments\notebook\arxiv_html_reference_workspace\metadata\arxiv_html_reference_meta.csv
  d:\Project\LSHBloomExperiments\notebook\arxiv_html_reference_workspace\metadata\arxiv_html_reference_meta.jsonl
  d:\Project\LSHBloomExperiments\notebook\arxiv_html_reference_workspace\jaccard\arxiv_html_vs_pymupdf_jaccard.csv
  d:\Project\LSHBloomExperiments\notebook\arxiv_html_reference_workspace\jaccard\arxiv_html_vs_pymupdf_summary.json

Summary:
{
  "n_html_ok": 1421,
  "n_pymupdf_ok": 1578,
  "n_common_valid": 1421,
  "mean_jaccard_html_vs_pymupdf": 0.6010847460169353,
  "median_jaccard_html_vs_pymupdf": 0.6059850374064838,
  "p10_jaccard_html_vs_pymupdf": 0.4610846812559467,
  "p90_jaccard_html_vs_pymupdf": 0.7405607476635514
}


## 10. Ghi chú

- Nếu `mean Jaccard` quanh **0.55–0.80** thì thường là vùng khá ổn cho near-duplicate parser benchmark.
- Nếu thấp bất thường, xem kỹ:
  - HTML parser có fallback về `body` quá nhiều không
  - text HTML có dính references / footnotes / boilerplate không
  - PyMuPDF có mất công thức hoặc line breaks quá nhiều không

Bạn có thể mở 3–10 case thấp nhất ở cell inspect để xem nhiễu nằm ở đâu.
